In [2]:
from pathlib import Path

dir_evals = Path('docs/temp/evals')
if not dir_evals.exists(): print(f"Directory {dir_evals} does not exist.")

tools = ['chatgpt_deepresearch', 'manus_ai', 'discourse2draft']
file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

In [3]:
import re
from src.backend.ai.llms import getAIModel
from pydantic import BaseModel, Field
from langchain.output_parsers.fix import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from src.backend.ai.prompts import setPrompt
from src.backend.utils import Config
import pandas as pd

class StandardizeSectionHeadersSchema(BaseModel):
    '''
    Returns the mapping from the section header to standard section header
    '''
    mapping: dict[str, str] = Field(description='A mapping from the section header to standard section header')

def standardizeHeader(section_headers, section_headers_standard):

    llm = getAIModel(model_name='azure-gpt-4o')
    system_prompt = 'You are an AI assist designed to standardize section headers in a document'
    human_prompt = '''
                You will be provided with a document which is a list of section headers.
                You will also be provided with a list of standard section headers.
                Your task is to map each provided section header to a standard section header.
                If there is no mapping possible, map the section header to itself.

                <Section headers>
                {section_headers}
                </Section headers>

                <Standard section headers>
                {section_headers_standard}
                </Standard section headers>
                '''
    
    parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=StandardizeSectionHeadersSchema), 
                                llm=llm,
                                max_retries=Config.RETRY_COUNTER)
    
    prompt = setPrompt(system_prompt, human_prompt, parser)
        
    response = (prompt | llm | parser).invoke(input={'section_headers': section_headers, 'section_headers_standard': section_headers_standard})
    return dict(response)['mapping']

def extractSectionContent(file_path):
    
    with open(file_path) as fp:
        data = '\n'.join([line.strip() for line in fp.readlines()])
    
    output = re.findall(r'(#+ .+\n?)([^#]+)', data)
    return [(section.strip(), content.strip()) for section, content in output]

def getSectionAndContent(file_path, section_headers_standard):
    doc = extractSectionContent(file_path)
    doc_san = []
    for section, content in doc:
        if section.startswith('# '):
            doc_san.append(['# Title', section[2:]])
        elif section.startswith('## '):
            doc_san.append([section, content])
        else:
            doc_san[-1][1] += section + '\n' + content
    section_headers = [section for section, _ in doc_san]
    mapping = standardizeHeader(section_headers, section_headers_standard)
    return [(mapping.get(section, section), content) for section, content in doc_san]
        
def getSectionsForComparison(file_name):

    section_headers_standard = [item for item in extractSectionContent(dir_evals / 'prompts' / file_name) if not item[0].startswith('# ')]
    responses = {tool: dict(getSectionAndContent(dir_evals / tool / file_name, section_headers_standard)) for tool in tools}
    
    return pd.DataFrame(responses)


In [4]:
file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    df_sections = getSectionsForComparison(file_name)
    df_sections.to_csv(dir_evals / 'sections_to_compare' / f'{'.'.join(file_name.split('.')[:-1])}.csv')
    display(df_sections)

Processing "CRISPR-based editing for inherited blood disorders.md"...


Calling src.backend.ai.llms.getAIModel

2026-03-18 08:13:42 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-03-18 08:13:43 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-03-18 08:13:45 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-03-18 08:13:47 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


...

Calling src.backend.ai.llms.getAIModel

2026-03-18 08:14:16 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-03-18 08:14:16 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-03-18 08:14:18 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-03-18 08:14:20 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


,chatgpt_deepresearch,manus_ai,discourse2draft
# Title,Health Impacts of Per- and Polyfluoroalkyl Sub...,Health Impacts of Per- and Polyfluoroalkyl Sub...,Title: Health Impacts of Per- and Polyfluoroal...
## Abstract:,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFAS) con...
## Introduction & Chemical Background:,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFAS) are...
## Exposure Pathways & Toxicokinetics:,**Exposure Pathways:** Human exposure to PFASs...,### 2.1. Exposure Pathways\nHuman exposure to ...,Drinking water contamination has emerged as a ...
## System-Specific Health Impacts (The Core Analysis):,### Immunotoxicity\nOne of the most sensitive ...,The evaluation of health impacts is based on t...,PFAS exposure has been consistently linked to ...
## Mechanisms of Action:,Elucidating how PFASs cause the array of obser...,The biological plausibility of the observed he...,PFAS interactions with peroxisome proliferator...
"## The ""GenX"" Problem:","By the early 2000s, scientific and regulatory ...",The phase-out of PFOA and PFOS led to the intr...,GenX (perfluoro-2-methyl-3-oxahexanoic acid) w...
## Regulatory Context & Knowledge Gaps:,**Regulatory Context:** Confronted with mounti...,### 6.1. Regulatory Context\nRegulatory bodies...,In its recent proposal under the Safe Drinking...
## Conclusion:,PFASs have transitioned from scientific obscur...,The weight of evidence from both epidemiologic...,Mounting epidemiological data underscore that ...
## References,1. **American Bar Association (ABA)**. (2022)....,[1]: https://pmc.ncbi.nlm.nih.gov/articles/PMC...,"[1] Sukhram, S. D., Kim, J., Musovic, S., Anid..."


In [5]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers.fix import OutputFixingParser
from pydantic import BaseModel, Field
from src.backend.ai.prompts import setPrompt
from src.backend.ai.common import StateContentManager
from src.backend.utils import Config

class ScoreWithReasonSchema(BaseModel):
    '''
    Returns a score within 0 to 100 for a particular criterion and a statement supporting the score
    '''
    score: int = Field(description='Score within 0 to 100')
    reason: str = Field(description='A short statement supporting the score')

class RateContentSchema(BaseModel):
    '''
    Returns scores based on different criteria to rate the content of a structured document
    '''

    relevance: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Relevance of the content')
    continuity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Continuity of the content')
    non_repetitiveness: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Uniqueness (or non-repetitiveness) of the content')
    specificity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Specificity of the content')
    #relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

# ---------------------------------------------------------------------------
class RateContent:

    rate_content_system_prompt = '''\
    You will be provided a manuscript section header with or without previously written content summary on a specific topic. You are a scholarly reviewer with expertise in the corresponding topic area. Your task is review and rate the section content.
    
    <Instructions>
    - Rating must be an integer number within 0 (lowest) and 100 (highest).
    - Rating will base on the following criteria.
    **Relevance:** Is the content relevant to the provided section header and the summary (if provided).
    **Continuity:** Is the content writing style continuous with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Non-Repetitiveness:** Is the content devoid of repetitiveness compared with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Specificity:** Is the content devoid of hallucination.
    </Instructions>
    '''

    rate_content_human_prompt = '''
    <Previous Content Summary>
    {content_pre}
    </Previous Content Summary>

    <Current Section Header>
    {current_section}
    </Current Section Header>

    <Content>
    {content}
    </Content>
    
    <Instructions>
    - Read the Previous Content Summary. 
    - Review the provided content based on the Current Section Header
    - Provide the output in the following format.
    {format_instructions}
    
    - Output must be in JSON format with `json` tags.
    </Instructions>
    '''

    def __init__(self, llm):

        parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=RateContentSchema), 
                                             llm=llm,
                                             max_retries=Config.RETRY_COUNTER)
        
        self.rate_content_prompt = setPrompt(self.rate_content_system_prompt, self.rate_content_human_prompt, parser)
        
        self.rate_content_chain = self.rate_content_prompt | llm | parser


    def __call__(self, state: StateContentManager):
        '''LLM generates reports from a given outline'''
        
        response = self.rate_content_chain.invoke(input={'content_pre': state['content_pre'],
                                                             'current_section': state['current_section'],
                                                             'content': state['content']})
        try:
            content = dict(response)
        except:
            raise Exception(f'GenerateContent response does not have content, response: {response}')

        return {'content': content, 'steps': ['Rate Content']}
    
from langgraph.graph import START, StateGraph
from src.backend.ai.llms import getAIModel
from src.backend.ai.summarize import Summarize
from typing import Literal

# -----------------------------------------------------------------------
def check_if_summary_needed(
        state: StateContentManager,
    ) -> Literal['Summarize', 'Rate Content']:
        if len(state.get('content_pre').split()) > 500:
            return 'Summarize'
        return 'Rate Content'

# -----------------------------------------------------------------------
class ContentWriterArchitecture:
     
     def __init__(self, model_name, temperature):
        llm = getAIModel(model_name=model_name, temperature=temperature)

        # Define a new graph
        workflow = StateGraph(state_schema=StateContentManager)

        # Define the (single) node in the graph
        workflow.add_node("Summarize", Summarize(llm=llm, input_field='content_pre'))
        workflow.add_node("Rate Content", RateContent(llm=llm))
        #workflow.add_node("Add Citations", AddCitations(llm))

        workflow.add_conditional_edges(START, check_if_summary_needed)
        workflow.add_edge("Summarize", "Rate Content")

        self.agent = workflow.compile()


In [11]:
import tqdm
import pandas as pd

# relevance, continuity, uniqueness, relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

from pathlib import Path

def formatRating(rating, tool):
    rating_dict = {}
    for criterion in rating:
        rating_dict[f'{tool} {criterion} (score)'] = rating[criterion].score
        rating_dict[f'{tool} {criterion} (reason)'] = rating[criterion].reason

    return rating_dict

for llm in ['gemini-3.1-pro']:

    agent = ContentWriterArchitecture(model_name=llm, temperature=0).agent

    for file_name in file_names:
        if file_name.startswith('CRISPR'): continue
        print(f'Processing "{file_name} with {llm} AI agent"...')
        section_sets = pd.read_csv(dir_evals / 'sections_to_compare' / f'{'.'.join(file_name.split('.')[:-1])}.csv', index_col=0)
        content_pre_sets = ['' for _ in section_sets]
        rating_responses = {}
        for index, row in tqdm.tqdm(section_sets.iterrows()):
            
            section_header = row.name
            print(section_header)
            
            rating_respons_section_wise = {}
            for i, tool in enumerate(tools):
                
                print(tool)

                content = row[tool] if not pd.isna(row[tool]) else ''

                if content.strip() == '': continue
                
                rating_response = agent.invoke({'current_section': section_header, 'content': content, 'content_pre': content_pre_sets[i]})['content']
                rating_respons_section_wise |= formatRating(rating_response, tool)

                content_pre_sets[i] += f'{section_header}\n\n{content}'

            rating_responses[section_header] = rating_respons_section_wise

        rating_responses = pd.DataFrame(rating_responses)

        rating_score = rating_responses.loc[rating_responses.index.str.endswith('(score)')]
        rating_reason = rating_responses.loc[rating_responses.index.str.endswith('(reason)')]

        rating_score = rating_score.rename(index=lambda x:x.replace(' (score)', ''))
        rating_reason = rating_reason.rename(index=lambda x:x.replace(' (reason)', ''))

        (dir_evals / 'scores' / llm).mkdir(parents=False, exist_ok=True)
        (dir_evals / 'reasons' / llm).mkdir(parents=False, exist_ok=True)

        rating_score.to_csv(dir_evals / 'scores' / llm / f'{Path(file_name).stem}.csv', index=True)
        rating_reason.to_csv(dir_evals / 'reasons' / llm / f'{Path(file_name).stem}.csv', index=True)

Calling src.backend.ai.llms.getAIModel

2026-03-19 08:21:29 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-03-19 08:21:29 [INFO] Calling src.backend.ai.prompts.setPrompt


Calling src.backend.ai.prompts.setPrompt

...

Calling src.backend.ai.common.extractLLMResponse

2026-03-19 08:41:36 [INFO] Calling src.backend.ai.common.extractLLMResponse
2026-03-19 08:41:56 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-03-19 08:42:01 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


discourse2draft


Calling src.backend.ai.common.extractLLMResponse

2026-03-19 08:42:01 [INFO] Calling src.backend.ai.common.extractLLMResponse
2026-03-19 08:42:20 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-03-19 08:42:32 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
10it [10:14, 61.41s/it]


In [6]:
import pandas as pd
import plotly.express as px

file_names_full = {
    'CRISPR-based editing for inherited blood disorders.md': 'CRISPR-based editing for inherited blood disorders',
    'Phthalates Toxicity.md': 'Phthalates Toxicity',
    'PFAS.md': 'Health Impacts of Per- and Polyfluoroalkyl Substances (PFASs): A Comprehensive Review'
}
feature_names = {'relevance': 'Relevance', 'continuity': 'Continuity', 'non_repetitiveness': 'Non-Repetitiveness', 'specificity': 'Specificity'}
tool_names = {'chatgpt_deepresearch': 'ChatGPT DeepResearch', 'manus_ai': 'Manus AI', 'discourse2draft': 'Discourse2Draft'}
evaluators = ['azure-o3', 'gemini-3.1-pro']

for i, file_name in enumerate(file_names):
    rating_scores = pd.concat([pd.read_csv(dir_evals / 'scores' / evaluator / f'{Path(file_name).stem}.csv', index_col=0) for evaluator in evaluators]).groupby(level=0).mean()
    rating_scores = rating_scores.rename(index=lambda x:x.replace(' (score)', ''))
    rating_scores[['Tools', 'Features']] = rating_scores.index.to_series().str.split(' ', expand=True)
    rating_scores = rating_scores.melt(id_vars=['Tools', 'Features'], var_name='Section Name', value_name='Score')
    rating_scores = rating_scores.loc[rating_scores['Section Name'] != '# Title']
    rating_scores['Features'] = rating_scores['Features'].map(feature_names)
    rating_scores['Tools'] = rating_scores['Tools'].map(tool_names)
    rating_scores = rating_scores.query('`Section Name` != "## References" or Features != "Continuity"')
    fig = px.bar(rating_scores, x='Section Name', y='Score', color='Tools', barmode='group', facet_col='Features', title='Content Evaluation Scores by Section and Tool', color_discrete_sequence=['#046b99', "#00a6d2", "#9bdaf1"])
    fig.update_xaxes(title='', tickangle=45)
    for annot in fig.layout.annotations:
        annot.text = annot.text.split('=')[1]
    if i == 0:
        fig.update_layout(title=file_names_full[file_name], template='plotly_white', title_x=0.5, legend={'y':1.5, 'x':0, 'orientation':'h'}, width=1200, height=600)
    else:
        fig.update_layout(title=file_names_full[file_name], template='plotly_white', title_x=0.5, showlegend=False, width=1200, height=600)

    fig.show()

In [8]:
rating_reasons = pd.read_csv(dir_evals / 'reasons' / 'azure-o3' / f'{Path(file_names[0]).stem}.csv', index_col=0)
rating_reasons = rating_reasons.rename(index=lambda x:x.replace(' (score)', ''))
rating_reasons

,# Title,## Executive Summary,## Molecular Mechanisms of Action,## Clinical Landscape: Hemoglobinopathies,"## The ""Ex Vivo"" Protocol & Technical Challenges",## Future Horizons: In Vivo Delivery,## Commercial & Ethical Analysis,## Key Terminology,## References
chatgpt_deepresearch relevance,"The content provides a concise, descriptive ph...",The content concisely synthesizes the current ...,Content focuses precisely on molecular mechani...,The section header calls for the clinical land...,The section focuses squarely on the ex-vivo HS...,Content focuses precisely on in-vivo delivery ...,The section directly addresses commercial (cos...,The section header calls for key terminology; ...,The section header is 'References' and the con...
chatgpt_deepresearch continuity,There is no prior text to create discontinuity...,"With only a title provided previously, the wri...",Seamlessly expands on the executive summary’s ...,"Tone, depth, citation style, and focus on exa-...","Tone, technical depth, and citation style are ...","Writing style, technical depth, and structure ...","Tone, depth, and terminology are consistent wi...",Although the presentation switches from narrat...,The reference list aligns with the citations u...
chatgpt_deepresearch non_repetitiveness,With only a single sentence serving as the tit...,"No prior narrative exists to duplicate, and wi...",Adds new mechanistic detail not present in the...,While new quantitative details and comparative...,"Some material (busulfan toxicity, p53 concerns...","Material introduces new concepts (LNPs, VLPs, ...","Little to no overlap with earlier material, wh...",Most definitions restate information already p...,A reference list inherently avoids narrative r...
chatgpt_deepresearch specificity,"The title is specific in scope, clearly identi...","Mentions of exa-cel approval, CLIMB-111/121 tr...",Technical explanations are largely accurate an...,"Clinical data, trial identifiers, and mechanis...","Descriptions of mobilization, editing efficien...","Discusses concrete delivery platforms, cites r...","Provides concrete price figures, examples of e...","Definitions are accurate, succinct, and free o...","While many citations appear legitimate, severa..."
manus_ai relevance,"The section header is ""# Title"" and the provid...",Content succinctly summarizes key points direc...,Content focuses precisely on molecular mechani...,Focuses on clinical trials and outcomes for SC...,Content directly outlines the ex vivo workflow...,The section focuses on in vivo delivery strate...,"The section focuses on cost, pricing models, g...",Glossary entries correspond directly to key mo...,The list is clearly a bibliography that corres...
manus_ai continuity,"No previous content summary was provided, so t...","Although only a title was previously provided,...",Writing style and technical depth follow natur...,Smooth transition from mechanistic discussion ...,"Writing style, technical depth, and citation f...","Tone, level of detail, and citation style (num...","Writing style (use of sub-headings, technical ...","Uses concise, technical language and formattin...","Numbering, citation style, and topical scope f..."
manus_ai non_repetitiveness,"With only a single sentence, there is no repet...",No earlier detailed content is available for c...,Adds mechanistic detail not present in the sum...,Introduces new clinical data not covered earli...,Adds new procedural and risk-mitigation inform...,"Introduces new concepts (targeted LNPs, AAV6) ...",Introduces new commercial and ethical consider...,While the information condenses material alrea...,Each entry is unique; there is no redundant li...
manus_ai specificity,The title is precise and accurately indicates ...,"Provides concrete, accurate details (e.g., Exa...",Mechanistic descriptions (BCL11A enhancer disr...,"Cites concrete trials (CLIMB-111/121), endpoin...","Details are largely accurate (mobilization, Bu...","Provides accurate, field-relevant details abou.